In [2]:
from IPython.core.debugger import prompt
from langgraph.graph import StateGraph , START,END
from typing import TypedDict , Literal
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
load_dotenv()

C:\Users\Archi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [8]:
import os
model = ChatGroq(api_key=os.getenv("API_KEY")
                 ,temperature=0.0,
                 model = "llama-3.3-70b-versatile")




In [3]:
class sentiment(BaseModel):
    sentiment: Literal["negative","positive"]


In [4]:
structured_model= model.with_structured_output(sentiment)

In [6]:
promt= "tell the sentiment in this custumer -this is a good product"
ans = structured_model.invoke(promt)
print(ans)


sentiment='positive'


In [3]:
class review_state(TypedDict):
     review : str
     sentiment : Literal["negative","positive"]
     diagnosis : dict
     response : str

In [17]:
def check_sentiment(state:review_state )->Literal["negative_diagnosis","positive_response"]:
     if state['review'] == "negative":
         return "negative_diagnosis"
     else:
         return "positive_response"
def positive_response(state:review_state ):
     promt = "thank the customer for the positive review"
     response = structured_model.invoke(promt)
     return {response:response}
def negative_diagnosis(state:review_state ):
     promt = "this is a negative review /n return tone urgency and issue type "

     response = structured_model2.invoke(promt)
     return {response:response.model_dump()}
def negative_response(state:review_state ):
    diagnosis = state['diagnosis']
    prompt = """you are a support model you need to give resolution /n/n on from these tone  {diagnosis['tone']} issue type {diagnosis['issue_type'], and urgency{diagnosis['urgency']}}"""
    response = structured_model.invoke(prompt)
    return {'response':response}


In [21]:
def sentiment_guess(state:review_state )->Literal["negative","positive"]:

    prompt=("this is the review form the customer {state['review']} tell if its negative or positve")
    response = structured_model.invoke(prompt)

In [9]:
structured_model2 = model.with_structured_output(diagnosis)


In [6]:
class diagnosis(BaseModel):
    tone : Literal['angry','frustrated','disappointed','calm'] = Field(description="Tone of the diagnosis")
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')

In [19]:
graph = StateGraph(review_state)

In [ ]:
graph.add_node("check sentiment",sentiment_guess)
graph.add_node("run diagnosis",negative_diagnosis)
graph.add_node("positve_response",positive_response)
graph.add_node("negative_response",negative_response)
graph.add_edge("")


In [20]:
graph.compile()

ValueError: Graph must have an entrypoint: add at least one edge from START to another node